# 🏥 Multi-Dataset Healthcare Integration Pipeline

## Overview
This notebook performs **complete preprocessing** and **cross-dataset integration** of all healthcare datasets from:

| Folder | Datasets |
|--------|----------|
| `Data_Set/` | UCI Heart Disease, Chronic Kidney Disease (CKD), eICU (eLCU), NHANES, CDC Diabetes, Breast Cancer Coimbra |
| `ND/` | MIMIC-IV v3.1, eICU-CRD v2.0, PTB-XL ECG (large + feature), HAM10000 Skin Lesion, Heart Failure EHR |

## Pipeline Stages
1. **Environment Setup** — Install libraries, configure paths, verify files
2. **Data_Set Preprocessing** — Load, clean, encode, impute, standardize original 6 datasets
3. **ND Preprocessing** — Load and preprocess 5 new large-scale clinical datasets
4. **Cross-Dataset Integration** — Unified schema, domain grouping, full merged output
5. **Quality Validation** — Missing values, distributions, bias audits, visualizations
6. **Export** — Save all preprocessed CSVs + integrated datasets + metadata JSON

---
## 🔧 Stage 1: Environment Setup

In [ ]:
# Stage 1: Install required packages
import subprocess, sys
required = ["pandas", "numpy", "matplotlib", "seaborn", "scikit-learn", "openpyxl"]
for pkg in required:
    try:
        __import__(pkg if pkg != "scikit-learn" else "sklearn")
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])
print("All packages ready.")


In [ ]:
import os, warnings, ast, json, gc, io
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from typing import Optional
from sklearn.impute import SimpleImputer
from matplotlib.gridspec import GridSpec
from datetime import datetime

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 140)
print("Libraries loaded.")


In [ ]:
BASE     = Path(r"e:/Data_Set")
DS_DIR   = BASE / "Data_Set"
ND_DIR   = BASE / "ND"
OUT_DIR  = BASE / "integrated_output"
PP_DIR   = OUT_DIR / "preprocessed"
DOM_DIR  = OUT_DIR / "by_domain"
PLOT_DIR = OUT_DIR / "plots"

for d in [OUT_DIR, PP_DIR, DOM_DIR, PLOT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print(f"BASE    : {BASE}")
print(f"DS_DIR  : {DS_DIR}")
print(f"ND_DIR  : {ND_DIR}")
print(f"OUT_DIR : {OUT_DIR}")


In [ ]:

PATHS = {
    "heart_cleveland"   : DS_DIR / "UCI_HeartDisease" / "processed.cleveland.csv",
    "heart_hungarian"   : DS_DIR / "UCI_HeartDisease" / "processed.hungarian.csv",
    "heart_switzerland" : DS_DIR / "UCI_HeartDisease" / "processed.switzerland.csv",
    "heart_va"          : DS_DIR / "UCI_HeartDisease" / "processed.va.csv",
    "ckd"               : DS_DIR / "Chronic_Kidney_Disease" / "Chronic_Kidney_Disease" / "chronic_kidney_disease_full.csv",
    "eicu_patient"      : DS_DIR / "eLCU" / "patient.csv",
    "eicu_diagnosis"    : DS_DIR / "eLCU" / "diagnosis.csv",
    "eicu_lab"          : DS_DIR / "eLCU" / "lab.csv",
    "nhanes"            : DS_DIR / "National Center for health Statistics" / "NHANES_combined.csv",
    "cdc_diabetes"      : DS_DIR / "CDC diabetes" / "diabetes_binary_health_indicators_BRFSS2015.csv",
    "breast_cancer"     : DS_DIR / "breast+cancer+coimbra_UCL" / "dataR2.csv",
    "mimic_patients"    : ND_DIR / "mimic-iv-3.1" / "hosp" / "patients.csv.gz",
    "mimic_admissions"  : ND_DIR / "mimic-iv-3.1" / "hosp" / "admissions.csv.gz",
    "mimic_icustays"    : ND_DIR / "mimic-iv-3.1" / "icu"  / "icustays.csv.gz",
    "eicu2_patient"     : ND_DIR / "eicu-collaborative-research-database-2.0" / "patient.csv.gz",
    "eicu2_diagnosis"   : ND_DIR / "eicu-collaborative-research-database-2.0" / "diagnosis.csv.gz",
    "eicu2_lab"         : ND_DIR / "eicu-collaborative-research-database-2.0" / "lab.csv.gz",
    "ptbxl_db"          : ND_DIR / "ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.0.3" / "ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.0.3" / "ptbxl_database.csv",
    "ptbxl_scp"         : ND_DIR / "ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.0.3" / "ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.0.3" / "scp_statements.csv",
    "ham10000"          : ND_DIR / "dataverse_files" / "HAM10000_metadata",
    "hf_ehr"            : ND_DIR / "hospitalized-patients-with-heart-failure-integrating-electronic-healthcare-records-and-external-outcome-data-1.2" / "hospitalized-patients-with-heart-failure-integrating-electronic-healthcare-records-and-external-outcome-data-1.2" / "dat.csv",
}

print("Dataset Path Verification:")
print("=" * 65)
for name, path in PATHS.items():
    exists = path.exists()
    size   = f"{path.stat().st_size/1e6:.1f} MB" if exists else "---"
    print(f"  [{'OK' if exists else 'MISSING':7s}] {name:25s}  {size:>10s}")


---
## 🔵 Stage 2: Data_Set Folder — Preprocessing

### 2.1 UCI Heart Disease (Cleveland, Hungarian, Switzerland, VA)

In [ ]:
HEART_COLUMNS = ['age','sex','cp','trestbps','chol','fbs','restecg','thalach',
                 'exang','oldpeak','slope','ca','thal','target_raw']

heart_files = ['heart_cleveland', 'heart_hungarian', 'heart_switzerland', 'heart_va']
heart_frames = []
for key in heart_files:
    path = PATHS[key]
    if path.exists():
        df = pd.read_csv(path, header=None, names=HEART_COLUMNS, na_values='?')
        df['source_site'] = key.replace('heart_', '')
        heart_frames.append(df)
        print(f'  Loaded {key}: {len(df)} rows')
    else:
        print(f'  WARNING: {key} not found')

if heart_frames:
    heart_raw = pd.concat(heart_frames, ignore_index=True)
    print(f'\nCombined Heart Disease: {heart_raw.shape}')
    print(heart_raw.head(3))
else:
    heart_raw = None
    print('WARNING: No heart disease files found')

In [ ]:
def preprocess_heart(df: pd.DataFrame) -> pd.DataFrame:
    """Preprocess UCI Heart Disease data."""
    out = df.copy()
    # Binary target: 0 = no disease, 1+ = disease
    out['target'] = (out['target_raw'] > 0).astype(int)
    out.drop(columns=['target_raw'], inplace=True, errors='ignore')

    # Age groups
    out['age_group'] = pd.cut(out['age'], bins=[0, 40, 60, 75, 120],
                              labels=['young', 'middle', 'senior', 'elderly'],
                              include_lowest=True)

    # Gender: 1=Male, 0=Female
    out['gender'] = out['sex'].map({1: 'Male', 0: 'Female'}).fillna('Unknown')

    # Impute numeric columns
    num_cols = out.select_dtypes(include=np.number).columns.difference(['target'])
    if len(num_cols) > 0:
        out[num_cols] = SimpleImputer(strategy='median').fit_transform(out[num_cols])

    out['dataset_source'] = 'UCI_HeartDisease'
    out['domain'] = 'cardiovascular'
    print(f'  Heart Disease preprocessed: {out.shape} | target rate: {out["target"].mean():.3f}')
    return out

if heart_raw is not None:
    heart_pp = preprocess_heart(heart_raw)
    heart_pp.head(3)
else:
    heart_pp = None

In [ ]:
# -- Heart Disease EDA Visualization --
if heart_pp is not None:
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    fig.suptitle('UCI Heart Disease', fontsize=13, fontweight='bold')

    heart_pp['target'].value_counts().plot.bar(ax=axes[0], color=['steelblue', 'coral'])
    axes[0].set_title('Target Distribution')
    axes[0].set_xticklabels(['No Disease', 'Disease'], rotation=0)

    axes[1].hist(heart_pp['age'].dropna(), bins=20, color='steelblue', edgecolor='white')
    axes[1].set_title('Age Distribution')

    heart_pp['gender'].value_counts().plot.bar(ax=axes[2], color=['royalblue', 'salmon'])
    axes[2].set_title('Gender Distribution')
    axes[2].set_xticklabels(axes[2].get_xticklabels(), rotation=0)

    plt.tight_layout()
    plt.savefig(PLOT_DIR / 'heart_disease_eda.png', bbox_inches='tight')
    plt.show()
    print('Heart Disease EDA saved.')

### 2.2 Chronic Kidney Disease (CKD)

In [ ]:
CKD_COLUMNS = ['age','bp','sg','al','su','rbc','pc','pcc','ba','bgr','bu','sc',
               'sod','pot','hemo','pcv','wc','rc','htn','dm','cad','appet','pe','ane','class']

CKD_NOMINAL_MAP = {
    'rbc':   {'normal': 0, 'abnormal': 1},
    'pc':    {'normal': 0, 'abnormal': 1},
    'pcc':   {'present': 1, 'notpresent': 0},
    'ba':    {'present': 1, 'notpresent': 0},
    'htn':   {'yes': 1, 'no': 0},
    'dm':    {'yes': 1, 'no': 0, ' yes': 1, '\tyes': 1, '\tno': 0},
    'cad':   {'yes': 1, 'no': 0, '\tno': 0},
    'appet': {'good': 0, 'poor': 1},
    'pe':    {'yes': 1, 'no': 0},
    'ane':   {'yes': 1, 'no': 0},
    'class': {'ckd': 1, 'notckd': 0, 'ckd\t': 1},
}

ckd_path = PATHS['ckd']
if ckd_path.exists():
    # ARFF-aware loader
    _raw = ckd_path.read_text(encoding='utf-8', errors='replace').splitlines()
    _arff_cols, _data_start = [], None
    for _i, _ln in enumerate(_raw):
        _s = _ln.strip().lower()
        if _s.startswith('@attribute'):
            _parts = _ln.strip().split()
            _col = _parts[1].strip("'\"")
            _arff_cols.append(_col)
        elif _s == '@data':
            _data_start = _i + 1
            break

    _norm = {'wbcc': 'wc', 'rbcc': 'rc'}
    _final_cols = [_norm.get(c, c) for c in _arff_cols]

    if _data_start is not None:
        _txt = '\n'.join(_raw[_data_start:])
        ckd_raw = pd.read_csv(
            io.StringIO(_txt), header=None, names=_final_cols,
            na_values=['?'], on_bad_lines='skip'
        )
    else:
        ckd_raw = pd.read_csv(
            ckd_path, comment='%', header=None, names=CKD_COLUMNS,
            na_values=['?'], on_bad_lines='skip'
        )

    ckd_raw.replace('?', np.nan, inplace=True)
    print(f'CKD raw shape: {ckd_raw.shape}')
    print(f'Columns: {ckd_raw.columns.tolist()}')
    ckd_raw.head(3)
else:
    ckd_raw = None
    print(f'WARNING: CKD file not found at {ckd_path}')

In [ ]:
def preprocess_ckd(df: pd.DataFrame) -> pd.DataFrame:
    """Preprocess Chronic Kidney Disease data."""
    out = df.copy()

    # Map nominal columns
    for col, mapping in CKD_NOMINAL_MAP.items():
        if col in out.columns:
            out[col] = out[col].astype(str).str.strip().str.lower().map(
                {str(k).lower(): v for k, v in mapping.items()}
            )
            out[col] = pd.to_numeric(out[col], errors='coerce')

    # Target from 'class' column
    if 'class' in out.columns:
        out['target'] = out['class']
        out.drop(columns=['class'], inplace=True, errors='ignore')
    else:
        out['target'] = np.nan

    # Convert numeric columns
    for col in out.columns:
        if col not in ['target']:
            out[col] = pd.to_numeric(out[col], errors='coerce')

    # Age groups
    if 'age' in out.columns:
        out['age_group'] = pd.cut(out['age'], bins=[0, 40, 60, 75, 120],
                                  labels=['young', 'middle', 'senior', 'elderly'],
                                  include_lowest=True)
    out['gender'] = 'Unknown'  # CKD dataset does not have gender

    # Impute
    num_cols = out.select_dtypes(include=np.number).columns.difference(['target'])
    if len(num_cols) > 0:
        out[num_cols] = SimpleImputer(strategy='median').fit_transform(out[num_cols])

    out['dataset_source'] = 'CKD_UCI'
    out['domain'] = 'renal'
    print(f'  CKD preprocessed: {out.shape} | target rate: {out["target"].mean():.3f}')
    return out

if ckd_raw is not None:
    ckd_pp = preprocess_ckd(ckd_raw)
    ckd_pp.head(3)
else:
    ckd_pp = None

In [ ]:
# -- CKD Visualization --
if ckd_pp is not None:
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    fig.suptitle('Chronic Kidney Disease', fontsize=13, fontweight='bold')

    ckd_pp['target'].value_counts().plot.bar(ax=axes[0], color=['steelblue', 'coral'])
    axes[0].set_title('Target Distribution')
    axes[0].set_xticklabels(['Not CKD', 'CKD'], rotation=0)

    axes[1].hist(ckd_pp['age'].dropna(), bins=20, color='steelblue', edgecolor='white')
    axes[1].set_title('Age Distribution')

    if 'hemo' in ckd_pp.columns:
        axes[2].hist(ckd_pp['hemo'].dropna(), bins=20, color='teal', edgecolor='white')
        axes[2].set_title('Hemoglobin Distribution')

    plt.tight_layout()
    plt.savefig(PLOT_DIR / 'ckd_eda.png', bbox_inches='tight')
    plt.show()

### 2.3 eICU (Original — eLCU folder)

In [ ]:
def preprocess_eicu_original() -> Optional[pd.DataFrame]:
    """Preprocess original eICU dataset from eLCU folder."""
    patient_path = PATHS['eicu_patient']
    diag_path    = PATHS['eicu_diagnosis']
    lab_path     = PATHS['eicu_lab']

    if not patient_path.exists():
        print('WARNING: eICU patient.csv not found -- skipping')
        return None

    print('  [eICU orig] Loading patient table...')
    pat = pd.read_csv(patient_path)
    print(f'    Patient records: {len(pat)} | columns: {pat.shape[1]}')

    if 'age' in pat.columns:
        pat['age'] = pat['age'].replace('> 89', 90).pipe(pd.to_numeric, errors='coerce')
    if 'gender'    in pat.columns: pat['gender']    = pat['gender'].str.strip().str.title()
    if 'ethnicity' in pat.columns: pat['ethnicity'] = pat['ethnicity'].str.strip().str.title()

    if 'hospitaldischargestatus' in pat.columns:
        pat['target'] = (pat['hospitaldischargestatus'].str.lower() == 'expired').astype(int)

    if 'age' in pat.columns:
        pat['age_group'] = pd.cut(pat['age'], bins=[0, 40, 60, 75, 120],
                                  labels=['young', 'middle', 'senior', 'elderly'], include_lowest=True)

    if diag_path.exists():
        print('    Merging diagnosis features...')
        diag = pd.read_csv(diag_path)
        diag_feat = diag.groupby('patientunitstayid').agg(
            diagnosis_count=('diagnosisid', 'count'),
            unique_dx_count=('diagnosisstring', 'nunique')
        ).reset_index()
        pat = pat.merge(diag_feat, on='patientunitstayid', how='left')
        pat[['diagnosis_count', 'unique_dx_count']] = pat[['diagnosis_count', 'unique_dx_count']].fillna(0)

    # lab.csv is optional and very large -- skip gracefully if absent
    if lab_path.exists():
        print('    Merging lab features...')
        try:
            lab = pd.read_csv(lab_path, usecols=['patientunitstayid', 'labname', 'labresult'])
            lab['labresult'] = pd.to_numeric(lab['labresult'], errors='coerce')
            top_labs  = lab['labname'].value_counts().head(15).index
            lab_pivot = lab[lab['labname'].isin(top_labs)].pivot_table(
                index='patientunitstayid', columns='labname', values='labresult', aggfunc='mean'
            ).reset_index()
            lab_pivot.columns = ['patientunitstayid'] + [
                f'lab_{c.lower().replace(" ","_")}' for c in lab_pivot.columns[1:]]
            pat = pat.merge(lab_pivot, on='patientunitstayid', how='left')
            print(f'    Lab features added: {lab_pivot.shape[1]-1}')
        except Exception as _e:
            print(f'    WARNING: Lab merge failed ({_e}) -- skipping')
    else:
        print('    Lab file not found -- skipping')

    miss_pct = pat.isnull().mean()
    pat.drop(columns=miss_pct[miss_pct > 0.6].index.tolist(), inplace=True)
    num_cols = pat.select_dtypes(include=np.number).columns.difference(['patientunitstayid', 'target'])
    if len(num_cols) > 0:
        pat[num_cols] = SimpleImputer(strategy='median').fit_transform(pat[num_cols])

    pat['dataset_source'] = 'eICU_original'
    pat['domain']         = 'icu'
    print(f'    -> shape: {pat.shape} | mortality: {pat["target"].mean():.3f}')
    return pat

eicu_pp = preprocess_eicu_original()
if eicu_pp is not None:
    print(f'\nOK eICU original preprocessed: {eicu_pp.shape}')
    eicu_pp.head(3)

### 2.4 NHANES (National Center for Health Statistics)

In [ ]:
def preprocess_nhanes() -> Optional[pd.DataFrame]:
    """Preprocess NHANES 2021-2023 data."""
    nhanes_path = PATHS['nhanes']
    if not nhanes_path.exists():
        print('WARNING: NHANES_combined.csv not found')
        return None

    print('  [NHANES] Loading...')
    df = pd.read_csv(nhanes_path, low_memory=False)
    print(f'    Raw shape: {df.shape}')

    # Gender: RIAGENDR or Gender column  (1=Male, 2=Female)
    for gcol in ['Gender', 'RIAGENDR']:
        if gcol in df.columns:
            df['gender'] = df[gcol].map({1: 'Male', 2: 'Female'}).fillna('Unknown')
            break
    else:
        df['gender'] = 'Unknown'

    # Age
    for acol in ['Age_Years', 'RIDAGEYR', 'age']:
        if acol in df.columns:
            df['age'] = pd.to_numeric(df[acol], errors='coerce')
            break

    df['age_group'] = pd.cut(df['age'], bins=[0, 40, 60, 75, 120],
                             labels=['young', 'middle', 'senior', 'elderly'],
                             include_lowest=True)

    # Target: create from diabetes indicators if available
    if 'target' not in df.columns:
        # Use glycohemoglobin > 6.5 or fasting glucose > 126 as diabetes proxy
        target_found = False
        for col in ['Glycohemoglobin_Pct', 'LBXGH']:
            if col in df.columns:
                df['target'] = (pd.to_numeric(df[col], errors='coerce') >= 6.5).astype(int)
                target_found = True
                break
        if not target_found:
            df['target'] = 0

    # Impute numeric columns
    num_cols = df.select_dtypes(include=np.number).columns.difference(['target', 'Patient_ID'])
    if len(num_cols) > 0:
        df[num_cols] = SimpleImputer(strategy='median').fit_transform(df[num_cols])

    df['dataset_source'] = 'NHANES'
    df['domain'] = 'population_health'
    print(f'    -> shape: {df.shape} | target rate: {df["target"].mean():.3f}')
    return df

nhanes_pp = preprocess_nhanes()
if nhanes_pp is not None:
    print(f'\nOK NHANES preprocessed: {nhanes_pp.shape}')

### 2.5 CDC Diabetes (BRFSS 2015)

In [ ]:
def preprocess_cdc_diabetes() -> Optional[pd.DataFrame]:
    """Preprocess CDC Diabetes BRFSS 2015."""
    path = PATHS['cdc_diabetes']
    if not path.exists():
        print('WARNING: CDC diabetes file not found')
        return None

    print('  [CDC Diabetes] Loading...')
    df = pd.read_csv(path)
    print(f'    Raw shape: {df.shape}')

    # Target
    if 'Diabetes_binary' in df.columns:
        df['target'] = df['Diabetes_binary'].astype(int)
    elif 'Diabetes_012' in df.columns:
        df['target'] = (df['Diabetes_012'] > 0).astype(int)

    # Gender: Sex column (0=Female, 1=Male in BRFSS)
    if 'Sex' in df.columns:
        df['gender'] = df['Sex'].map({1: 'Male', 0: 'Female'}).fillna('Unknown')

    # Age: BRFSS uses age categories 1-13
    if 'Age' in df.columns:
        age_map = {1: 22, 2: 27, 3: 32, 4: 37, 5: 42, 6: 47, 7: 52,
                   8: 57, 9: 62, 10: 67, 11: 72, 12: 77, 13: 80}
        df['age'] = df['Age'].map(age_map)
    df['age_group'] = pd.cut(df['age'], bins=[0, 40, 60, 75, 120],
                             labels=['young', 'middle', 'senior', 'elderly'],
                             include_lowest=True)

    df['dataset_source'] = 'CDC_Diabetes'
    df['domain'] = 'metabolic'
    print(f'    -> shape: {df.shape} | target rate: {df["target"].mean():.3f}')
    return df

cdc_pp = preprocess_cdc_diabetes()
if cdc_pp is not None:
    print(f'\nOK CDC Diabetes preprocessed: {cdc_pp.shape}')

### 2.6 Breast Cancer Coimbra (UCI)

In [ ]:
def preprocess_breast_cancer() -> Optional[pd.DataFrame]:
    """Preprocess Breast Cancer Coimbra dataset."""
    path = PATHS['breast_cancer']
    if not path.exists():
        print('WARNING: Breast cancer file not found')
        return None

    print('  [Breast Cancer] Loading...')
    df = pd.read_csv(path)
    print(f'    Raw shape: {df.shape}')

    # Target: Classification (1=Healthy, 2=Patient)
    if 'Classification' in df.columns:
        df['target'] = (df['Classification'] == 2).astype(int)

    # All female dataset
    df['gender'] = 'Female'

    # Age
    if 'Age' in df.columns:
        df['age'] = pd.to_numeric(df['Age'], errors='coerce')

    df['age_group'] = pd.cut(df['age'], bins=[0, 40, 60, 75, 120],
                             labels=['young', 'middle', 'senior', 'elderly'],
                             include_lowest=True)

    # Impute
    num_cols = df.select_dtypes(include=np.number).columns.difference(['target', 'Classification'])
    if len(num_cols) > 0:
        df[num_cols] = SimpleImputer(strategy='median').fit_transform(df[num_cols])

    df['dataset_source'] = 'BreastCancer_Coimbra'
    df['domain'] = 'oncology'
    print(f'    -> shape: {df.shape} | target rate: {df["target"].mean():.3f}')
    return df

bc_pp = preprocess_breast_cancer()
if bc_pp is not None:
    print(f'\nOK Breast Cancer preprocessed: {bc_pp.shape}')

In [ ]:
# -- Data_Set Summary --
print('Data_Set Folder -- Preprocessing Summary')
print('=' * 65)
ds_datasets = {
    'UCI_HeartDisease'     : heart_pp,
    'CKD_UCI'              : ckd_pp,
    'eICU_original'        : eicu_pp,
    'NHANES'               : nhanes_pp,
    'CDC_Diabetes'         : cdc_pp,
    'BreastCancer_Coimbra' : bc_pp,
}
for name, df in ds_datasets.items():
    if df is not None:
        print(f'  {name:30s}: {len(df):>8,} rows x {df.shape[1]:>3} cols')
    else:
        print(f'  {name:30s}: FAILED / NOT FOUND')
print(f'\nTotal: {sum(len(d) for d in ds_datasets.values() if d is not None):,} records')

---
## 🟢 Stage 3: ND Folder — New Dataset Preprocessing

### 3.1 MIMIC-IV v3.1 (Hospital + ICU EHR)

In [ ]:
def preprocess_mimic_iv() -> Optional[pd.DataFrame]:
    """Preprocess MIMIC-IV v3.1 dataset."""
    pat_path = PATHS['mimic_patients']
    adm_path = PATHS['mimic_admissions']
    icu_path = PATHS['mimic_icustays']

    if not pat_path.exists():
        print('WARNING: MIMIC patients.csv.gz not found')
        return None

    print('  [MIMIC-IV] Loading patients...')
    patients = pd.read_csv(pat_path)
    print(f'    Patients: {len(patients):,}')

    # Gender mapping
    patients['gender'] = patients['gender'].str.strip().str.upper().map(
        {'M': 'Male', 'F': 'Female'}
    ).fillna('Unknown')

    # Age
    patients['age'] = pd.to_numeric(patients['anchor_age'], errors='coerce')
    patients['age_group'] = pd.cut(patients['age'], bins=[0, 40, 60, 75, 120],
                                   labels=['young', 'middle', 'senior', 'elderly'],
                                   include_lowest=True)

    # Admissions
    if adm_path.exists():
        print('    Merging admissions...')
        adm = pd.read_csv(adm_path)
        # Aggregate admissions per patient
        adm_agg = adm.groupby('subject_id').agg(
            n_admissions=('hadm_id', 'nunique'),
            hospital_expire_flag=('hospital_expire_flag', 'max'),
            insurance_first=('insurance', 'first'),
            race_first=('race', 'first'),
        ).reset_index()
        patients = patients.merge(adm_agg, on='subject_id', how='left')
        print(f'    After admissions merge: {patients.shape}')

    # ICU stays count
    if icu_path.exists():
        print('    Merging ICU stays...')
        icu = pd.read_csv(icu_path)
        icu_agg = icu.groupby('subject_id').agg(
            n_icu_stays=('stay_id', 'nunique'),
            total_icu_los=('los', 'sum'),
            avg_icu_los=('los', 'mean'),
        ).reset_index()
        patients = patients.merge(icu_agg, on='subject_id', how='left')
        patients[['n_icu_stays', 'total_icu_los', 'avg_icu_los']] =             patients[['n_icu_stays', 'total_icu_los', 'avg_icu_los']].fillna(0)
        print(f'    After ICU merge: {patients.shape}')

    # Target = hospital mortality
    if 'hospital_expire_flag' in patients.columns:
        patients['target'] = patients['hospital_expire_flag'].fillna(0).astype(int)
    else:
        patients['target'] = 0

    patients['dataset_source'] = 'MIMIC_IV'
    patients['domain'] = 'icu'
    print(f'    -> shape: {patients.shape} | mortality: {patients["target"].mean():.3f}')
    return patients

mimic_pp = preprocess_mimic_iv()
if mimic_pp is not None:
    print(f'\nOK MIMIC-IV preprocessed: {mimic_pp.shape}')
    mimic_pp.head(3)

### 3.2 eICU-CRD v2.0 (New Large-Scale ICU Dataset)

In [ ]:
def preprocess_eicu_v2() -> Optional[pd.DataFrame]:
    """Preprocess eICU-CRD v2.0 dataset."""
    pat_path  = PATHS['eicu2_patient']
    diag_path = PATHS['eicu2_diagnosis']
    lab_path  = PATHS['eicu2_lab']

    if not pat_path.exists():
        print('WARNING: eICU-CRD v2 patient.csv.gz not found')
        return None

    print('  [eICU-CRD v2] Loading patient table...')
    pat = pd.read_csv(pat_path)
    print(f'    Patient records: {len(pat):,}')

    # Age
    if 'age' in pat.columns:
        pat['age'] = pat['age'].replace('> 89', 90).pipe(pd.to_numeric, errors='coerce')
    if 'gender' in pat.columns:
        pat['gender'] = pat['gender'].str.strip().str.title()
    if 'ethnicity' in pat.columns:
        pat['ethnicity'] = pat['ethnicity'].str.strip().str.title()

    # Target: hospital mortality
    if 'hospitaldischargestatus' in pat.columns:
        pat['target'] = (pat['hospitaldischargestatus'].str.lower() == 'expired').astype(int)
    else:
        pat['target'] = 0

    pat['age_group'] = pd.cut(pat['age'], bins=[0, 40, 60, 75, 120],
                              labels=['young', 'middle', 'senior', 'elderly'],
                              include_lowest=True)

    # Diagnosis features
    if diag_path.exists():
        print('    Merging diagnosis features...')
        diag = pd.read_csv(diag_path)
        diag_feat = diag.groupby('patientunitstayid').agg(
            diagnosis_count=('diagnosisid', 'count'),
            unique_dx_count=('diagnosisstring', 'nunique')
        ).reset_index()
        pat = pat.merge(diag_feat, on='patientunitstayid', how='left')
        pat[['diagnosis_count', 'unique_dx_count']] = pat[['diagnosis_count', 'unique_dx_count']].fillna(0)

    # Lab features (optional -- very large)
    if lab_path.exists():
        print('    Merging lab features (sampling for speed)...')
        try:
            lab = pd.read_csv(lab_path, usecols=['patientunitstayid', 'labname', 'labresult'])
            lab['labresult'] = pd.to_numeric(lab['labresult'], errors='coerce')
            top_labs = lab['labname'].value_counts().head(15).index
            lab_pivot = lab[lab['labname'].isin(top_labs)].pivot_table(
                index='patientunitstayid', columns='labname', values='labresult', aggfunc='mean'
            ).reset_index()
            lab_pivot.columns = ['patientunitstayid'] + [
                f'lab_{c.lower().replace(" ","_")}' for c in lab_pivot.columns[1:]]
            pat = pat.merge(lab_pivot, on='patientunitstayid', how='left')
            print(f'    Lab features added: {lab_pivot.shape[1]-1}')
        except Exception as _e:
            print(f'    WARNING: Lab merge failed ({_e}) -- skipping')

    # Drop very sparse columns and impute
    miss_pct = pat.isnull().mean()
    pat.drop(columns=miss_pct[miss_pct > 0.6].index.tolist(), inplace=True)
    num_cols = pat.select_dtypes(include=np.number).columns.difference(['patientunitstayid', 'target'])
    if len(num_cols) > 0:
        pat[num_cols] = SimpleImputer(strategy='median').fit_transform(pat[num_cols])

    pat['dataset_source'] = 'eICU_CRD_v2'
    pat['domain'] = 'icu'
    print(f'    -> shape: {pat.shape} | mortality: {pat["target"].mean():.3f}')
    return pat

eicu2_pp = preprocess_eicu_v2()
if eicu2_pp is not None:
    print(f'\nOK eICU-CRD v2 preprocessed: {eicu2_pp.shape}')
    eicu2_pp.head(3)

### 3.3 PTB-XL ECG Dataset

In [ ]:
def preprocess_ptbxl() -> Optional[pd.DataFrame]:
    """Preprocess PTB-XL ECG dataset (metadata only, not waveforms)."""
    db_path  = PATHS['ptbxl_db']
    scp_path = PATHS['ptbxl_scp']

    if not db_path.exists():
        print('WARNING: ptbxl_database.csv not found')
        return None

    print('  [PTB-XL] Loading database...')
    df = pd.read_csv(db_path)
    print(f'    Records: {len(df):,}')

    # Gender: 0=Male, 1=Female
    if 'sex' in df.columns:
        df['gender'] = df['sex'].map({0: 'Male', 1: 'Female'}).fillna('Unknown')

    # Age
    df['age'] = pd.to_numeric(df['age'], errors='coerce')
    df['age_group'] = pd.cut(df['age'], bins=[0, 40, 60, 75, 120],
                             labels=['young', 'middle', 'senior', 'elderly'],
                             include_lowest=True)

    # Parse SCP codes to create target (any abnormal finding)
    if 'scp_codes' in df.columns:
        # Load SCP statements for classification
        if scp_path.exists():
            scp = pd.read_csv(scp_path, index_col=0)
            # Abnormal = diagnostic class is not NORM
            def is_abnormal(scp_str):
                try:
                    codes = ast.literal_eval(scp_str)
                    for code in codes:
                        if code != 'NORM' and codes[code] > 50:
                            return 1
                    return 0
                except:
                    return 0
            df['target'] = df['scp_codes'].apply(is_abnormal)
        else:
            df['target'] = 0
    else:
        df['target'] = 0

    # Extract useful signal quality features
    for col in ['height', 'weight', 'heart_axis', 'validated_by_human',
                'baseline_drift', 'static_noise', 'burst_noise',
                'electrodes_problems', 'extra_beats', 'pacemaker', 'strat_fold']:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')

    # Drop non-useful columns
    drop_cols = ['filename_lr', 'filename_hr', 'report', 'initial_autogenerated_report',
                 'scp_codes', 'recording_date', 'nurse', 'site', 'device',
                 'infarction_stadium1', 'infarction_stadium2', 'validated_by', 'second_opinion']
    df.drop(columns=[c for c in drop_cols if c in df.columns], inplace=True)

    df['dataset_source'] = 'PTB_XL_ECG'
    df['domain'] = 'ecg'
    print(f'    -> shape: {df.shape} | abnormal rate: {df["target"].mean():.3f}')
    return df

ptbxl_pp = preprocess_ptbxl()
if ptbxl_pp is not None:
    print(f'\nOK PTB-XL preprocessed: {ptbxl_pp.shape}')
    ptbxl_pp.head(3)

### 3.4 HAM10000 Skin Lesion Dataset

In [ ]:
def preprocess_ham10000() -> Optional[pd.DataFrame]:
    """Preprocess HAM10000 skin lesion dataset."""
    path = PATHS['ham10000']
    if not path.exists():
        print('WARNING: HAM10000_metadata not found')
        return None

    print('  [HAM10000] Loading metadata...')
    df = pd.read_csv(path)
    print(f'    Records: {len(df):,}')

    # Gender
    if 'sex' in df.columns:
        df['gender'] = df['sex'].str.strip().str.lower().map(
            {'male': 'Male', 'female': 'Female'}
        ).fillna('Unknown')

    # Age
    df['age'] = pd.to_numeric(df['age'], errors='coerce')
    df['age_group'] = pd.cut(df['age'], bins=[0, 40, 60, 75, 120],
                             labels=['young', 'middle', 'senior', 'elderly'],
                             include_lowest=True)

    # Target: melanoma (mel) = 1, all others = 0
    if 'dx' in df.columns:
        df['target'] = (df['dx'] == 'mel').astype(int)

    # One-hot encode diagnosis type and localization
    if 'dx' in df.columns:
        dx_dummies = pd.get_dummies(df['dx'], prefix='dx')
        df = pd.concat([df, dx_dummies], axis=1)

    if 'localization' in df.columns:
        loc_dummies = pd.get_dummies(df['localization'], prefix='loc')
        df = pd.concat([df, loc_dummies], axis=1)

    if 'dx_type' in df.columns:
        dxtype_dummies = pd.get_dummies(df['dx_type'], prefix='dxtype')
        df = pd.concat([df, dxtype_dummies], axis=1)

    df['dataset_source'] = 'HAM10000'
    df['domain'] = 'dermatology'
    print(f'    -> shape: {df.shape} | melanoma rate: {df["target"].mean():.3f}')
    return df

ham_pp = preprocess_ham10000()
if ham_pp is not None:
    print(f'\nOK HAM10000 preprocessed: {ham_pp.shape}')
    ham_pp.head(3)

### 3.5 Heart Failure EHR Dataset

In [ ]:
def preprocess_heart_failure_ehr() -> Optional[pd.DataFrame]:
    """Preprocess hospitalized heart failure EHR dataset."""
    path = PATHS['hf_ehr']
    if not path.exists():
        print('WARNING: Heart Failure EHR dat.csv not found')
        return None

    print('  [Heart Failure EHR] Loading...')
    df = pd.read_csv(path, low_memory=False)
    print(f'    Raw shape: {df.shape}')

    # Drop identifier columns
    drop_cols = ['Unnamed: 0', 'inpatient.number', 'DestinationDischarge',
                 'admission.ward', 'admission.way', 'occupation', 'discharge.department']
    df.drop(columns=[c for c in drop_cols if c in df.columns], inplace=True)

    # Gender
    if 'gender' in df.columns:
        df['gender'] = df['gender'].astype(str).str.strip().str.lower().map(
            {'male': 'Male', 'female': 'Female', '1': 'Male', '0': 'Female',
             '1.0': 'Male', '0.0': 'Female', 'm': 'Male', 'f': 'Female'}
        ).fillna('Unknown')
    else:
        df['gender'] = 'Unknown'

    # Target: in-hospital mortality or readmission
    if 'target' not in df.columns:
        # Use outcome.during.hospitalization or re.admission if available
        target_found = False
        for tcol in ['outcome.during.hospitalization', 're.admission.within.30.days',
                     'outcome', 'mortality']:
            if tcol in df.columns:
                df['target'] = pd.to_numeric(df[tcol], errors='coerce').fillna(0).astype(int)
                target_found = True
                break
        if not target_found:
            df['target'] = 0

    # Age: not available in this dataset
    df['age'] = np.nan
    df['age_group'] = 'Unknown'

    # Scale numeric columns
    num_cols = df.select_dtypes(include=np.number).columns.difference(['target'])
    if len(num_cols) > 0:
        from sklearn.preprocessing import StandardScaler
        scaler = StandardScaler()
        scaled = pd.DataFrame(
            scaler.fit_transform(df[num_cols].fillna(df[num_cols].median())),
            columns=[f'{c}_scaled' for c in num_cols],
            index=df.index
        )
        df = pd.concat([df, scaled], axis=1)

    df['dataset_source'] = 'HeartFailure_EHR'
    df['domain'] = 'cardiovascular'
    print(f'    -> shape: {df.shape} | target rate: {df["target"].mean():.3f}')
    return df

hf_pp = preprocess_heart_failure_ehr()
if hf_pp is not None:
    print(f'\nOK Heart Failure EHR preprocessed: {hf_pp.shape}')
    hf_pp.head(3)

In [ ]:
# -- ND Summary --
print('ND Folder -- Preprocessing Summary')
print('=' * 65)
nd_datasets = {
    'MIMIC_IV'         : mimic_pp,
    'eICU_CRD_v2'      : eicu2_pp,
    'PTB_XL_ECG'       : ptbxl_pp,
    'HAM10000'         : ham_pp,
    'HeartFailure_EHR' : hf_pp,
}
for name, df in nd_datasets.items():
    if df is not None:
        print(f'  {name:30s}: {len(df):>8,} rows x {df.shape[1]:>3} cols')
    else:
        print(f'  {name:30s}: FAILED / NOT FOUND')
print(f'\nTotal: {sum(len(d) for d in nd_datasets.values() if d is not None):,} records')

---
## 🔗 Stage 4: Cross-Dataset Integration

### 4.1 Unified Common Schema Definition

In [ ]:
COMMON_SCHEMA = ["dataset_source", "domain", "target", "age", "age_group", "gender"]

def _std_gender(series, mapping=None):
    s = series.astype(str).str.strip().str.lower()
    if mapping:
        lmap = {str(k).lower(): v for k, v in mapping.items()}
        out  = s.map(lmap)
        mask = out.isna()
        out[mask] = s[mask].map(
            lambda x: "Male"   if x in ("m","male","man","1","1.0") else
                      "Female" if x in ("f","female","woman","0","0.0","2","2.0") else
                      "Unknown"
        )
        return out
    return s.map(
        lambda x: "Male"   if x in ("m","male","man","1","1.0") else
                  "Female" if x in ("f","female","woman","0","0.0","2","2.0") else
                  "Unknown"
    )

def _bin_age(s):
    return (pd.cut(s, bins=[0,40,60,75,120],
                   labels=["young","middle","senior","elderly"],
                   include_lowest=True)
            .astype(str).replace("nan","Unknown"))

# Registry: how each preprocessed CSV maps to the unified schema
REGISTRY = {
    "heart_disease_preprocessed.csv" : dict(source="UCI_HeartDisease",    domain="cardiovascular",
        target_col="target",   age_col="age",       gender_col="sex",
        gender_map={"1":"Male","1.0":"Male","0":"Female","0.0":"Female"}),
    "ckd_preprocessed.csv"           : dict(source="CKD_UCI",             domain="renal",
        target_col="class", target_map={"ckd":1,"notckd":0,"1":1,"0":0,"1.0":1,"0.0":0},
        age_col="age",       gender_fixed="Unknown"),
    "eicu_original_preprocessed.csv" : dict(source="eICU_original",       domain="icu",
        target_col="target",   age_col="age",       gender_col="gender"),
    "nhanes_preprocessed.csv"        : dict(source="NHANES",              domain="population_health",
        target_col="target",   age_col="Age_Years", gender_col="Gender",
        gender_map={"1":"Male","1.0":"Male","2":"Female","2.0":"Female"}),
    "cdc_diabetes_preprocessed.csv"  : dict(source="CDC_Diabetes",        domain="metabolic",
        target_col="target",   age_col="Age",       gender_col="Sex",
        gender_map={"1":"Male","1.0":"Male","0":"Female","0.0":"Female"}),
    "breast_cancer_preprocessed.csv" : dict(source="BreastCancer_Coimbra",domain="oncology",
        target_col="Classification", target_map={"1":0,"1.0":0,"2":1,"2.0":1},
        age_col="Age",       gender_fixed="Female"),
    "mimic_iv_preprocessed.csv"      : dict(source="MIMIC_IV",            domain="icu",
        target_col="hospital_expire_flag", age_col="age", gender_col="gender"),
    "eicu_crd_v2_preprocessed.csv"   : dict(source="eICU_CRD_v2",         domain="icu",
        target_col="target",   age_col="age",       gender_col="gender"),
    "ptbxl_preprocessed.csv"         : dict(source="PTB_XL_ECG",          domain="ecg",
        target_col="target",   age_col="age",       gender_col="sex",
        gender_map={"0":"Male","0.0":"Male","1":"Female","1.0":"Female"}),
    "ham10000_preprocessed.csv"      : dict(source="HAM10000",            domain="dermatology",
        target_col="target",   age_col="age",       gender_col="sex"),
    "heart_failure_ehr_preprocessed.csv": dict(source="HeartFailure_EHR", domain="cardiovascular",
        target_col="death.within.28.days", age_col=None, gender_col="gender"),
}

def harmonize_to_schema(df, cfg):
    out = df.copy()
    out["dataset_source"] = cfg["source"]
    out["domain"]         = cfg["domain"]

    # target
    t = cfg.get("target_col")
    if t and t in out.columns:
        tm = cfg.get("target_map")
        if tm:
            out["target"] = out[t].astype(str).str.strip().str.lower().map(
                {str(k).lower():v for k,v in tm.items()})
        else:
            out["target"] = pd.to_numeric(out[t], errors="coerce")
        if t != "target":
            out.drop(columns=[t], inplace=True, errors="ignore")
    elif "target" not in out.columns:
        out["target"] = np.nan
    valid = out["target"].notna()
    out.loc[valid & (out["target"] > 1), "target"] = 1
    out["target"] = pd.to_numeric(out["target"], errors="coerce")

    # age
    a = cfg.get("age_col")
    if a and a in out.columns:
        out["age"] = pd.to_numeric(
            out[a].astype(str).str.replace("> 89","90",regex=False), errors="coerce")
        if a != "age":
            out.drop(columns=[a], inplace=True, errors="ignore")
    elif "age" not in out.columns:
        out["age"] = np.nan

    # age_group
    if "age_group" not in out.columns:
        out["age_group"] = _bin_age(out["age"])
    else:
        out["age_group"] = out["age_group"].astype(str).replace("nan","Unknown")

    # gender
    gf = cfg.get("gender_fixed")
    if gf:
        out["gender"] = gf
    else:
        g = cfg.get("gender_col")
        if g and g in out.columns:
            out["gender"] = _std_gender(out[g], cfg.get("gender_map"))
            if g != "gender":
                out.drop(columns=[g], inplace=True, errors="ignore")
        elif "gender" not in out.columns:
            out["gender"] = "Unknown"

    front = [c for c in COMMON_SCHEMA if c in out.columns]
    other = [c for c in out.columns if c not in front]
    print(f"  [{cfg['source']:<30}] -> {out.shape} | "
          f"target={out['target'].mean():.3f} | "
          f"age_ok={out['age'].notna().mean()*100:.0f}%")
    return out[front + other]

print("Schema helpers & registry defined.")


In [ ]:
print("Harmonizing all datasets...")
print("=" * 72)

harmonized = {}
for fname, cfg in REGISTRY.items():
    path = PP_DIR / fname
    if not path.exists():
        print(f"  [SKIP] {cfg['source']} — preprocessed file not found")
        continue
    df = pd.read_csv(path, low_memory=False)
    harmonized[cfg["source"]] = harmonize_to_schema(df, cfg)

print(f"\n{len(harmonized)}/{len(REGISTRY)} datasets harmonized successfully")
if len(harmonized) < len(REGISTRY):
    missing = [cfg["source"] for f,cfg in REGISTRY.items() if cfg["source"] not in harmonized]
    print(f"WARNING - missing: {missing}")


### 4.2 Domain-Level Integration

In [ ]:
domain_groups = {}
for name, df in harmonized.items():
    dom = df["domain"].iat[0]
    domain_groups.setdefault(dom, []).append((name, df))

print("Domain Groups:")
print("=" * 60)
for dom, items in sorted(domain_groups.items()):
    total  = sum(len(d) for _,d in items)
    labels = [f"{n}({len(d):,})" for n,d in items]
    print(f"  {dom:<25}: {total:>8,} records  <-  {', '.join(labels)}")


In [ ]:
def integrate_domain(datasets, domain_name):
    frames = [df for _,df in datasets]
    merged = pd.concat(frames, axis=0, ignore_index=True, sort=False)
    front  = [c for c in COMMON_SCHEMA if c in merged.columns]
    other  = [c for c in merged.columns if c not in front]
    merged = merged[front + other]
    print(f"  [{domain_name:<20}] -> {merged.shape} | missing: {merged.isnull().mean().mean()*100:.1f}%")
    return merged

print("Integrating by domain...")
print("=" * 60)
domain_dfs = {}
for dom, items in sorted(domain_groups.items()):
    domain_dfs[dom] = integrate_domain(items, dom)
print(f"\n{len(domain_dfs)} domain DataFrames created")


### 4.3 Full Cross-Dataset Integration (All Datasets)

In [ ]:
# ── Concatenate all datasets ──────────────────────────────────────────────────
print("Concatenating all datasets...")
all_frames      = list(harmonized.values())
full_integrated = pd.concat(all_frames, axis=0, ignore_index=True, sort=False)

front = [c for c in COMMON_SCHEMA if c in full_integrated.columns]
other = [c for c in full_integrated.columns if c not in front]
full_integrated = full_integrated[front + other]

print(f"Before imputation: {len(full_integrated):,} rows × {full_integrated.shape[1]} cols")
print(f"Overall NaN %    : {full_integrated.isnull().mean().mean()*100:.1f}%")

# ── Smart Imputation — fill every empty cell ──────────────────────────────────
print("\nApplying smart imputation to fill all empty cells...")
print("-" * 60)

nan_before = full_integrated.isnull().sum().sum()
print(f"  NaN cells before: {nan_before:,}")

# Schema text columns
for col in ["dataset_source", "domain"]:
    if col in full_integrated.columns:
        full_integrated[col] = full_integrated[col].fillna("Unknown")
for col in ["gender", "age_group"]:
    if col in full_integrated.columns:
        full_integrated[col] = full_integrated[col].replace({"nan":"Unknown","":" Unknown"}).fillna("Unknown")

# Numeric columns
num_cols  = full_integrated.select_dtypes(include=[np.number]).columns.tolist()
text_cols = [c for c in full_integrated.select_dtypes(exclude=[np.number]).columns
             if c not in COMMON_SCHEMA]

filled_total = 0
for col in num_cols:
    n_null = full_integrated[col].isna().sum()
    if n_null == 0:
        continue
    non_null = full_integrated[col].dropna()
    if len(non_null) == 0:
        fill_val = 0
    else:
        unique_vals = set(non_null.unique())
        is_binary   = unique_vals.issubset({0, 1, 0.0, 1.0})
        fill_val    = 0 if is_binary else float(non_null.median())
    full_integrated[col] = full_integrated[col].fillna(fill_val)
    filled_total += n_null

for col in text_cols:
    n_null = full_integrated[col].isna().sum()
    if n_null == 0:
        continue
    vals = full_integrated[col].dropna()
    fill_val = vals.mode().iloc[0] if len(vals) > 0 else "Unknown"
    full_integrated[col] = full_integrated[col].fillna(fill_val)
    filled_total += n_null

# Re-derive age_group for HeartFailure rows where age is still NaN
mask_unk = full_integrated["age_group"] == "Unknown"
if mask_unk.any():
    full_integrated.loc[mask_unk, "age_group"] = _bin_age(
        full_integrated.loc[mask_unk, "age"]).values
    full_integrated["age_group"] = full_integrated["age_group"].replace("nan","Unknown")

nan_after = full_integrated.isnull().sum().sum()
print(f"  NaN cells after : {nan_after:,}  (filled {filled_total:,} cells)")
print(f"\nAfter imputation: {len(full_integrated):,} rows × {full_integrated.shape[1]} cols")
print(f"Overall NaN %   : {full_integrated.isnull().mean().mean()*100:.2f}%  (target: 0.00%)")

print("\n" + "="*70)
print("FULL INTEGRATED DATASET — SUMMARY")
print("="*70)
print(f"  Total records  : {len(full_integrated):>12,}")
print(f"  Total features : {full_integrated.shape[1]:>12,}")
print()
print("  Records per dataset source:")
for ds, cnt in full_integrated["dataset_source"].value_counts().items():
    print(f"    {ds:<32} {cnt:>10,}")
print()
print("  Records per clinical domain:")
for dom, cnt in full_integrated["domain"].value_counts().items():
    print(f"    {dom:<32} {cnt:>10,}")

# Verify sample looks clean
print("\nSample (3 rows × 12 cols):")
print(full_integrated.head(3).iloc[:, :12].to_string(index=False))


---
## 📊 Stage 5: Quality Validation & Visualization

In [ ]:
summary_rows = []
for name, df in harmonized.items():
    t_rate = df["target"].mean() if df["target"].notna().any() else None
    summary_rows.append({
        "Dataset"     : name,
        "Domain"      : df["domain"].iat[0],
        "Records"     : len(df),
        "Columns"     : df.shape[1],
        "Missing %"   : round(df.isnull().mean().mean()*100, 1),
        "Target Rate" : round(t_rate, 3) if t_rate is not None else "N/A",
        "Age Avail %" : round(df["age"].notna().mean()*100, 1) if "age" in df.columns else 0,
    })

summary_df = pd.DataFrame(summary_rows).sort_values("Records", ascending=False)
print("Dataset Quality Summary (before imputation):")
print(summary_df.to_string(index=False))


In [ ]:
fig = plt.figure(figsize=(22, 18))
gs  = GridSpec(3, 2, figure=fig, hspace=0.38, wspace=0.3)

# 1. Records per dataset
ax1 = fig.add_subplot(gs[0, 0])
rec    = full_integrated["dataset_source"].value_counts()
colors = plt.cm.Set3(np.linspace(0, 1, len(rec)))
bars   = ax1.barh(range(len(rec)), rec.values, color=colors, edgecolor="white")
ax1.set_yticks(range(len(rec)))
ax1.set_yticklabels([t[:22] for t in rec.index], fontsize=8)
ax1.set_title("Records per Dataset", fontweight="bold")
ax1.set_xlabel("Count")
for bar, val in zip(bars, rec.values):
    ax1.text(bar.get_width()*1.01, bar.get_y()+bar.get_height()/2,
             f"{val:,}", va="center", fontsize=7)

# 2. Missing % per dataset (before imputation)
ax2 = fig.add_subplot(gs[0, 1])
miss_by_ds   = {n: df.isnull().mean().mean()*100 for n, df in harmonized.items()}
names_sorted = sorted(miss_by_ds, key=miss_by_ds.get, reverse=True)
vals         = [miss_by_ds[n] for n in names_sorted]
max_miss     = max(vals) + 1e-6 if vals else 1
colors2      = [plt.cm.RdYlGn_r(v / max_miss) for v in vals]
ax2.barh(range(len(names_sorted)), vals, color=colors2, edgecolor="white")
ax2.set_yticks(range(len(names_sorted)))
ax2.set_yticklabels([n[:22] for n in names_sorted], fontsize=8)
ax2.set_title("Missing Value % per Dataset (before imputation)", fontweight="bold")
ax2.set_xlabel("Missing %")

# 3. Records per domain pie
ax3 = fig.add_subplot(gs[1, 0])
dom_counts = full_integrated["domain"].value_counts()
ax3.pie(dom_counts.values, labels=dom_counts.index, autopct="%1.1f%%",
        colors=plt.cm.Pastel1(np.linspace(0, 1, len(dom_counts))), startangle=90)
ax3.set_title("Records by Clinical Domain", fontweight="bold")

# 4. Target rate per dataset
ax4 = fig.add_subplot(gs[1, 1])
trates = {n: df["target"].mean() for n, df in harmonized.items()
          if "target" in df.columns and df["target"].notna().any()}
ax4.barh(list(trates.keys()), list(trates.values()), color="coral", edgecolor="white")
ax4.set_title("Target Rate per Dataset", fontweight="bold")
ax4.set_xlabel("Rate (0-1)")
ax4.set_yticklabels([k[:22] for k in trates.keys()], fontsize=8)

# 5. Gender distribution
ax5 = fig.add_subplot(gs[2, 0])
gc = full_integrated["gender"].value_counts()
ax5.bar(gc.index, gc.values, color=["royalblue","salmon","lightgrey"][:len(gc)])
ax5.set_title("Gender Distribution (All Datasets)", fontweight="bold")
ax5.set_ylabel("Count")
for i, (_, v) in enumerate(gc.items()):
    ax5.text(i, v + gc.max()*0.01, f"{v:,}", ha="center", fontsize=9)

# 6. Age distribution
ax6 = fig.add_subplot(gs[2, 1])
age_data = full_integrated["age"].dropna()
ax6.hist(age_data, bins=50, color="steelblue", edgecolor="white", alpha=0.8)
ax6.set_title("Age Distribution (All Datasets)", fontweight="bold")
ax6.set_xlabel("Age (years)")
ax6.set_ylabel("Count")

fig.suptitle("Multi-Dataset Integration Dashboard", fontsize=16, fontweight="bold", y=1.01)
plt.savefig(PLOT_DIR / "dataset_overview.png", bbox_inches="tight", dpi=150)
plt.show()
print("Dashboard saved.")


In [ ]:
# -- Feature Overlap Matrix --
print('Feature Overlap Analysis Between Datasets')
print('=' * 60)

feature_sets = {name: set(df.columns) for name, df in harmonized.items()}
all_names    = list(feature_sets.keys())

overlap_matrix = pd.DataFrame(
    [[len(feature_sets[n1] & feature_sets[n2]) for n2 in all_names] for n1 in all_names],
    index=all_names, columns=all_names
)

fig, ax = plt.subplots(figsize=(12, 9))
short_names = [n[:18] for n in all_names]
sns.heatmap(overlap_matrix.values, xticklabels=short_names, yticklabels=short_names,
            ax=ax, annot=True, fmt='d', cmap='Blues', linewidths=0.3,
            cbar_kws={'label': 'Shared Columns'})
ax.set_title('Shared Feature Count Between Datasets', fontsize=13, fontweight='bold')
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right', fontsize=8)
ax.set_yticklabels(ax.get_yticklabels(), fontsize=8)
plt.tight_layout()
plt.savefig(PLOT_DIR / 'shared_features_heatmap.png', bbox_inches='tight')
plt.show()
print('Feature overlap heatmap saved.')

In [ ]:
# -- Bias / Fairness Audit: Target Rate by Gender --
print('Bias Audit: Target Rate by Gender per Dataset')
print('=' * 60)

bias_rows = []
for name, df in harmonized.items():
    if 'target' in df.columns and 'gender' in df.columns:
        valid_df = df[df['target'].notna() & (df['gender'] != 'Unknown')]
        if len(valid_df) > 0:
            grp = valid_df.groupby('gender', observed=False)['target'].agg(['mean', 'count'])
            for gender, row in grp.iterrows():
                if row['count'] > 10:
                    bias_rows.append({'Dataset': name[:18], 'Gender': gender,
                                      'Rate': row['mean'], 'Count': int(row['count'])})

if bias_rows:
    bias_df = pd.DataFrame(bias_rows)
    pivot = bias_df.pivot_table(values='Rate', index='Dataset', columns='Gender', aggfunc='mean')
    print(pivot.round(3).to_string())

    fig, ax = plt.subplots(figsize=(12, 6))
    pivot.plot.barh(ax=ax, color=['royalblue', 'salmon'])
    ax.set_title('Target Rate by Gender per Dataset', fontweight='bold')
    ax.set_xlabel('Target Rate')
    ax.legend(title='Gender')
    plt.tight_layout()
    plt.savefig(PLOT_DIR / 'bias_audit_gender.png', bbox_inches='tight')
    plt.show()
    print('Bias audit chart saved.')
else:
    print('No bias data available.')

In [ ]:
# -- Age-Target Relationship across Domains --
domains_with_data = [
    (dom, df) for dom, df in domain_dfs.items()
    if 'age' in df.columns and 'target' in df.columns and df['target'].notna().sum() > 10
    and df['age'].notna().sum() > 10
]

if domains_with_data:
    n_plots = min(len(domains_with_data), 4)
    fig, axes = plt.subplots(1, n_plots, figsize=(5*n_plots, 5))
    if n_plots == 1: axes = [axes]
    fig.suptitle('Age vs. Target Rate by Domain', fontsize=13, fontweight='bold')

    for i, (domain, df_d) in enumerate(domains_with_data[:n_plots]):
        df_clean = df_d.dropna(subset=['age','target']).copy()
        if len(df_clean) < 10:
            axes[i].text(0.5, 0.5, f'{domain}\nInsufficient data', ha='center', va='center',
                         transform=axes[i].transAxes)
            continue
        try:
            df_clean['age_bin'] = pd.cut(df_clean['age'], bins=10)
            rates = df_clean.groupby('age_bin', observed=False)['target'].mean()
            rates.plot(ax=axes[i], marker='o', color='coral')
            axes[i].set_title(f'{domain}', fontsize=10, fontweight='bold')
            axes[i].set_xlabel('Age Bin')
            axes[i].set_ylabel('Target Rate')
            axes[i].tick_params(axis='x', rotation=45, labelsize=7)
        except Exception as e:
            axes[i].text(0.5, 0.5, f'{domain}\nError: {e}', ha='center', va='center',
                         transform=axes[i].transAxes, fontsize=8)

    plt.tight_layout()
    plt.savefig(PLOT_DIR / 'age_target_by_domain.png', bbox_inches='tight')
    plt.show()
    print('Age-target chart saved.')
else:
    print('No domains with sufficient age+target data.')

In [ ]:
schema_cols_present = [c for c in COMMON_SCHEMA if c in full_integrated.columns]

# Use ORIGINAL harmonized data (before imputation) so we see true availability
avail_data = {}
for name, df in harmonized.items():
    avail_data[name] = {
        col: round(df[col].notna().sum() / len(df) * 100, 1)
        for col in schema_cols_present if col in df.columns
    }

avail_df = pd.DataFrame(avail_data).T.fillna(0)
fig, ax  = plt.subplots(figsize=(10, 7))
sns.heatmap(avail_df, ax=ax, cmap="YlGn", annot=True, fmt=".0f",
            linewidths=0.5, cbar_kws={"label":"% Available (non-null — before imputation)"},
            vmin=0, vmax=100)
ax.set_title("Schema Column Original Availability per Dataset (%)", fontsize=12, fontweight="bold")
ax.set_yticklabels(ax.get_yticklabels(), fontsize=9)
plt.tight_layout()
plt.savefig(PLOT_DIR / "schema_availability.png", bbox_inches="tight")
plt.show()
print("Heatmap saved.")


---
## 💾 Stage 6: Export Preprocessed & Integrated Data

In [ ]:
print("Saving individual preprocessed datasets...")
pp_dir = OUT_DIR / "preprocessed"
pp_dir.mkdir(exist_ok=True)

save_map = {
    "heart_disease_preprocessed.csv"    : heart_pp   if "heart_pp"   in dir() else None,
    "ckd_preprocessed.csv"              : ckd_pp     if "ckd_pp"     in dir() else None,
    "eicu_original_preprocessed.csv"    : eicu_pp    if "eicu_pp"    in dir() else None,
    "nhanes_preprocessed.csv"           : nhanes_pp  if "nhanes_pp"  in dir() else None,
    "cdc_diabetes_preprocessed.csv"     : cdc_pp     if "cdc_pp"     in dir() else None,
    "breast_cancer_preprocessed.csv"    : bc_pp      if "bc_pp"      in dir() else None,
    "mimic_iv_preprocessed.csv"         : mimic_pp   if "mimic_pp"   in dir() else None,
    "eicu_crd_v2_preprocessed.csv"      : eicu2_pp   if "eicu2_pp"   in dir() else None,
    "ptbxl_preprocessed.csv"            : ptbxl_pp   if "ptbxl_pp"   in dir() else None,
    "ham10000_preprocessed.csv"         : ham_pp     if "ham_pp"     in dir() else None,
    "heart_failure_ehr_preprocessed.csv": hf_pp      if "hf_pp"      in dir() else None,
}
for fname, df in save_map.items():
    if df is not None:
        p = pp_dir / fname
        df.to_csv(p, index=False)
        print(f"  {fname:<45} {df.shape}  {p.stat().st_size/1e3:>8.0f} KB")
    else:
        print(f"  {fname:<45} SKIPPED")


In [ ]:
print("\nSaving domain-integrated datasets (from imputed full data)...")
domain_dir = OUT_DIR / "by_domain"
domain_dir.mkdir(exist_ok=True)

for dom, items in sorted(domain_groups.items()):
    # Pull rows from imputed full dataset, keep only relevant columns
    member_cols = set(COMMON_SCHEMA)
    for _, orig_df in items:
        member_cols |= set(orig_df.columns)
    dom_df   = full_integrated[full_integrated["domain"] == dom]
    dom_df   = dom_df[[c for c in dom_df.columns if c in member_cols]]
    fname    = f"{dom}_integrated.csv"
    out_path = domain_dir / fname
    dom_df.to_csv(out_path, index=False)
    size     = out_path.stat().st_size / 1e3
    labels   = [f"{n}({len(d):,})" for n, d in items]
    print(f"  {fname:<35} {dom_df.shape}  {size:>8.0f} KB  <-  {', '.join(labels)}")


In [ ]:
print("\nSaving full integrated dataset (imputed — no empty cells)...")

# Full dataset
full_path = OUT_DIR / "full_integrated_dataset.csv"
full_integrated.to_csv(full_path, index=False)
size_mb   = full_path.stat().st_size / 1e6
print(f"  full_integrated_dataset.csv      -> {full_integrated.shape}  ({size_mb:.1f} MB)")

# Core (6 schema columns only)
core_cols = [c for c in COMMON_SCHEMA if c in full_integrated.columns]
core_path = OUT_DIR / "core_integrated_dataset.csv"
full_integrated[core_cols].to_csv(core_path, index=False)
core_mb   = core_path.stat().st_size / 1e6
print(f"  core_integrated_dataset.csv      -> {full_integrated[core_cols].shape}  ({core_mb:.1f} MB)")

# Summary schema (copy of core)
full_integrated[core_cols].to_csv(OUT_DIR / "integration_summary_schema.csv", index=False)
print(f"  integration_summary_schema.csv   -> {full_integrated[core_cols].shape}")

# Quick sanity check
remaining_nan = full_integrated.isnull().sum().sum()
print(f"\n  Remaining NaN cells in full file : {remaining_nan}  (should be 0)")


In [ ]:
meta = {
    "run_timestamp"     : datetime.now().isoformat(timespec="seconds"),
    "pipeline_version"  : "5.0",
    "total_records"     : int(len(full_integrated)),
    "total_features"    : int(full_integrated.shape[1]),
    "imputation_applied": True,
    "nan_after_imputation": int(full_integrated.isnull().sum().sum()),
    "datasets_integrated": list(harmonized.keys()),
    "domains"           : sorted(domain_groups.keys()),
    "common_schema"     : COMMON_SCHEMA,
    "per_dataset": {
        name: {
            "records"        : int(len(df)),
            "original_columns": int(df.shape[1]),
            "missing_pct_before_imputation":
                round(float(df.isnull().mean().mean() * 100), 2),
            "has_target"     : bool(df["target"].notna().any()),
            "target_rate"    : round(float(df["target"].mean()), 4)
                               if df["target"].notna().any() else None,
            "domain"         : str(df["domain"].iat[0]),
            "age_pct"        : round(float(df["age"].notna().mean()*100), 1)
                               if "age" in df.columns else 0.0,
            "gender_counts"  : df["gender"].value_counts().to_dict(),
        }
        for name, df in harmonized.items()
    },
}
meta_path = OUT_DIR / "integration_metadata.json"
with open(meta_path, "w", encoding="utf-8") as f:
    json.dump(meta, f, indent=2, ensure_ascii=False)
print(f"integration_metadata.json saved -> {meta_path}")


---
## 📑 Stage 7: Final Summary Report

In [ ]:
remaining = full_integrated.isnull().sum().sum()
print("=" * 78)
print("  MULTI-DATASET HEALTHCARE INTEGRATION — FINAL REPORT  (v5)")
print("=" * 78)
print(f"  Datasets integrated   : {len(harmonized)}")
print(f"  Clinical domains      : {len(domain_groups)}")
print(f"  Total patient records : {len(full_integrated):,}")
print(f"  Total feature columns : {full_integrated.shape[1]}")
print(f"  NaN cells remaining   : {remaining}  (0 = perfectly clean)")
print()
print(f"  {'Dataset':<30} {'Domain':<20} {'Records':>10} {'Cols':>5} {'Target%':>8}")
print("  " + "-" * 76)
for name, df in harmonized.items():
    dom  = df["domain"].iat[0]
    rate = f"{df['target'].mean()*100:.1f}%" if df["target"].notna().any() else "N/A"
    print(f"  {name:<30} {dom:<20} {len(df):>10,} {df.shape[1]:>5} {rate:>8}")
print("  " + "-" * 76)
print(f"  {'TOTAL':<30} {'':<20} {len(full_integrated):>10,} {full_integrated.shape[1]:>5}")
print()
print("  Output files:")
for p in [OUT_DIR / "full_integrated_dataset.csv",
          OUT_DIR / "core_integrated_dataset.csv",
          OUT_DIR / "integration_metadata.json"]:
    if p.exists():
        print(f"    {p.name:<45} {p.stat().st_size/1e6:.1f} MB")
dom_files = list(DOM_DIR.glob("*_integrated.csv"))
print(f"    by_domain/  -> {len(dom_files)} domain files")
print()
print("  PIPELINE COMPLETE — all empty cells filled, no commas without values.")
print("=" * 78)


---
## 👁️ Stage 8: View Full Integrated Data
Run cells below to see all 868,874 rows across all 11 datasets.

In [ ]:
import pandas as pd
import numpy as np

# Remove ALL display limits so you see the FULL data
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)
pd.set_option('display.float_format', '{:.4f}'.format)

print('Loading full integrated dataset...')
df = pd.read_csv(
    r'd:\Data_Set\integrated_output\full_integrated_dataset.csv',
    low_memory=False
)
print(f'Loaded: {len(df):,} rows  x  {df.shape[1]} columns')
print(f'Datasets: {sorted(df["dataset_source"].unique())}')
print(f'Domains : {sorted(df["domain"].unique())}')


### 8.1 — All Columns in the Dataset

In [ ]:
# Show every column name and data type
col_info = pd.DataFrame({
    'Column'  : df.columns,
    'Dtype'   : df.dtypes.values,
    'NonNull%': (df.notna().mean() * 100).round(1).values,
    'Sample'  : [str(df[c].dropna().iloc[0]) if df[c].notna().any() else 'N/A'
                 for c in df.columns]
})
print(f'Total columns: {len(col_info)}')
display(col_info)


### 8.2 — Records per Dataset (All 11)

In [ ]:
counts = df['dataset_source'].value_counts()
print('Records per Dataset:')
print('-' * 55)
for ds, cnt in counts.items():
    pct = cnt / len(df) * 100
    bar = '#' * int(pct / 2)
    print(f'  {ds:<30} {cnt:>8,}  ({pct:5.1f}%)  {bar}')
print('-' * 55)
print(f'  {"TOTAL":<30} {len(df):>8,}  (100.0%)')


### 8.3 — Records per Clinical Domain

In [ ]:
for dom, cnt in df['domain'].value_counts().items():
    pct = cnt / len(df) * 100
    bar = '#' * int(pct / 2)
    print(f'  {dom:<25} {cnt:>8,}  ({pct:5.1f}%)  {bar}')


### 8.4 — Full Data: All Rows for Every Dataset

In [ ]:
# Shows ALL rows for EACH dataset with only its real columns (no empty cols)
for ds in df['dataset_source'].unique():
    sub = df[df['dataset_source'] == ds].reset_index(drop=True)
    # Keep only columns that have more than 1 unique value for this dataset
    real_cols = (['dataset_source', 'domain', 'target', 'age', 'age_group', 'gender']
                 + [c for c in sub.columns
                    if c not in ['dataset_source','domain','target','age','age_group','gender']
                    and sub[c].nunique() > 1])
    sub = sub[real_cols]
    print()
    print('=' * 80)
    print(f'  DATASET: {ds}  |  {len(sub):,} rows  x  {len(real_cols)} columns')
    print('=' * 80)
    with pd.option_context('display.max_rows', None,
                           'display.max_columns', None,
                           'display.width', None):
        display(sub)


### 8.5 — Core Schema: All 868,874 Rows (6 Key Columns)

In [ ]:
# Full view of all records — 6 unified columns only
schema = ['dataset_source', 'domain', 'target', 'age', 'age_group', 'gender']
core   = df[schema].reset_index(drop=True)
print(f'Core dataset: {core.shape}')
with pd.option_context('display.max_rows', None,
                       'display.max_columns', None,
                       'display.width', None):
    display(core)


### 8.6 — Statistical Summary per Dataset

In [ ]:
summary_rows = []
for ds in df['dataset_source'].unique():
    sub = df[df['dataset_source'] == ds]
    summary_rows.append({
        'Dataset'       : ds,
        'Domain'        : sub['domain'].iat[0],
        'Rows'          : len(sub),
        'Columns'       : sub.shape[1],
        'Target_Rate_%' : round(sub['target'].mean() * 100, 2),
        'Age_Mean'      : round(sub['age'].mean(), 1),
        'Age_Min'       : round(sub['age'].min(), 1),
        'Age_Max'       : round(sub['age'].max(), 1),
        'Male_%'        : round((sub['gender'] == 'Male').mean() * 100, 1),
        'Female_%'      : round((sub['gender'] == 'Female').mean() * 100, 1),
        'Unknown_%'     : round((sub['gender'] == 'Unknown').mean() * 100, 1),
    })
stat_df = pd.DataFrame(summary_rows)
with pd.option_context('display.max_rows', None, 'display.max_columns', None,
                       'display.width', None):
    display(stat_df)


### 8.7 — Filter & View Any Domain (Change DOMAIN below)

In [ ]:
# Change this to any domain you want:
# Options: 'icu', 'cardiovascular', 'metabolic', 'ecg',
#          'dermatology', 'population_health', 'renal', 'oncology'
DOMAIN = 'cardiovascular'

subset    = df[df['domain'] == DOMAIN].reset_index(drop=True)
real_cols = (['dataset_source','domain','target','age','age_group','gender']
             + [c for c in subset.columns
                if c not in ['dataset_source','domain','target','age','age_group','gender']
                and subset[c].nunique() > 1])
subset    = subset[real_cols]
print(f'Domain [{DOMAIN}]: {len(subset):,} rows  x  {len(real_cols)} columns')
with pd.option_context('display.max_rows', None, 'display.max_columns', None,
                       'display.width', None):
    display(subset)


### 8.8 — Search Any Specific Dataset

In [ ]:
# Change DATASET_NAME to any of the 11 datasets:
# UCI_HeartDisease, CKD_UCI, eICU_original, NHANES, CDC_Diabetes,
# BreastCancer_Coimbra, MIMIC_IV, eICU_CRD_v2, PTB_XL_ECG, HAM10000, HeartFailure_EHR
DATASET_NAME = 'MIMIC_IV'

ds_df     = df[df['dataset_source'] == DATASET_NAME].reset_index(drop=True)
real_cols = [c for c in ds_df.columns if ds_df[c].nunique() > 1
             or c in ['dataset_source','domain','target','age','age_group','gender']]
ds_df     = ds_df[real_cols]
print(f'Dataset [{DATASET_NAME}]: {len(ds_df):,} rows  x  {len(real_cols)} columns')
with pd.option_context('display.max_rows', None, 'display.max_columns', None,
                       'display.width', None):
    display(ds_df)
